# 07 · OOP Part 2: Inheritance & Beyond

In Part 1 you learned to build classes, protect their data, and make objects that *contain* other objects ("has-a"). Now we make classes **related** to each other ("is-a"), and learn the tools that make your classes feel like built-in Python types.

**What's inside**

| # | Topic |
|---|---|
| 1 | Inheritance and `super()` |
| 2 | Method overriding |
| 3 | Polymorphism |
| 4 | Duck typing |
| 5 | Overriding vs overloading |
| 6 | `__eq__`: comparing objects |
| 7 | `__repr__`: the developer's view |
| 8 | `@property`: smarter getters and setters |
| 9 | `isinstance()` and guard checks |
| 10 | `@classmethod` and `@staticmethod` |
| 11 | Splitting code into files: modules and `__name__` |
| 12 | Practice problems |

---
## 1. Inheritance and `super()`

**Inheritance** lets a new class **reuse** everything from an existing class and then add or change things. The existing class is the **parent** (base / superclass); the new one is the **child** (derived / subclass).

Use it when there is an **"is-a"** relationship:
- a `Car` **is a** `Vehicle`
- a `Manager` **is an** `Employee`
- a `Pizza` **is a** `MenuItem`

```
class Child(Parent):
```

Inside the child's `__init__`, call **`super().__init__(...)`** to let the parent set up its part first. (`super()` means "my parent". It's `base(...)` in C# and `super(...)` in Java.)

> 💡 **Real life:** every animal eats and sleeps. A `Dog` *is an* `Animal`, so it gets eating and sleeping for free, and only adds what makes a dog special (barking).

### 📚 From class

In [1]:
class Vehicle:
    def __init__(self, brand, model):
        self.brand = brand
        self.model = model

    def __str__(self):
        return f"{self.brand} {self.model}"

    def honk(self):
        print("Beep beep!")


class Car(Vehicle):
    def __init__(self, brand, model, year):
        super().__init__(brand, model)      # parent sets brand and model
        self.year = year                     # child adds its own data

    def honk(self):                          # overrides the parent's version
        print("Honk honk!")


class Bike(Vehicle):
    def __init__(self, brand, model, color):
        super().__init__(brand, model)
        self.color = color

    def honk(self):
        super().honk()                       # run the parent's version first...
        print("it still goes Ring ring!")    # ...then add more


car1 = Car("Toyota", "Corolla", 2020)
bike1 = Bike("Trek", "Marlin 7", "red")

print(car1)              # __str__ was inherited from Vehicle
print(bike1)
car1.honk()
bike1.honk()

Toyota Corolla
Trek Marlin 7
Honk honk!
Beep beep!
it still goes Ring ring!


### 🆕 More examples

In [2]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def annual_pay(self):
        return self.salary * 12

    def __str__(self):
        return f"{self.name} (Rs {self.salary}/month)"


class Manager(Employee):
    def __init__(self, name, salary, team_size):
        super().__init__(name, salary)
        self.team_size = team_size

    def annual_pay(self):
        bonus = self.team_size * 10000         # managers earn a bonus per team member
        return super().annual_pay() + bonus     # reuse the parent's calculation


class Developer(Employee):
    def __init__(self, name, salary, language):
        super().__init__(name, salary)
        self.language = language


staff = [Employee("Ali", 50000), Manager("Sara", 90000, 4), Developer("Zain", 80000, "Python")]

for person in staff:
    print(person, "->", person.annual_pay())

Ali (Rs 50000/month) -> 600000
Sara (Rs 90000/month) -> 1120000
Zain (Rs 80000/month) -> 960000


`Developer` didn't define `annual_pay`, so it **inherited** the one from `Employee`. `Manager` defined its own, so that one is used.

### How does Python choose which method to run?
It looks in the object's own class first, then in the parent, then the grandparent… and uses the **first one it finds**.

In [3]:
print(Manager.__mro__)      # "method resolution order": the search path
print(isinstance(staff[1], Employee))
print(issubclass(Manager, Employee))

(<class '__main__.Manager'>, <class '__main__.Employee'>, <class 'object'>)
True
True


---
## 2. Method overriding

When a child defines a method with the **same name** as the parent's, the child's version **replaces** the parent's for child objects. This is **overriding**.

You can still reach the parent's version with `super().method()` when you want to *extend* instead of *replace* (like `Bike.honk` above).

| You want to… | Do |
|---|---|
| keep the parent's behaviour | don't define the method in the child |
| **replace** it completely | define it in the child |
| **extend** it | define it and call `super().method()` inside |

### 🆕 More examples

In [4]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"


class Dog(Animal):
    def speak(self):                          # replace
        return f"{self.name} says Woof!"


class Cat(Animal):
    def speak(self):                          # extend
        return super().speak() + " ...specifically, Meow."


class Fish(Animal):
    pass                                      # inherits everything


for pet in [Dog("Rex"), Cat("Tom"), Fish("Nemo")]:
    print(pet.speak())

Rex says Woof!
Tom makes a sound ...specifically, Meow.
Nemo makes a sound


---
## 3. Polymorphism

**Polymorphism** ("many forms") means: *the same method call does the right thing for different kinds of objects.* You write code that just calls `.speak()` or `.area()` and each object supplies its own behaviour. No giant `if type == ...` chains.

### 📚 From class

In [5]:
class Vehicle:
    def honk(self):
        print("Beep beep!")

class Bike(Vehicle):
    def honk(self):
        print("Ring ring!")

vehicles = [Vehicle(), Bike()]
for v in vehicles:
    v.honk()             # same call, different result

Beep beep!
Ring ring!


In [6]:
class Shape:
    def area(self):
        return 0

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return 3.14159 * self.radius ** 2

class Square(Shape):
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height
    def area(self):
        return 0.5 * self.base * self.height


shapes = [Circle(5), Square(4), Triangle(6, 3), Shape()]

for shape in shapes:
    print(f"{type(shape).__name__} area: {shape.area():.2f}")

Circle area: 78.54
Square area: 16.00
Triangle area: 9.00
Shape area: 0.00


### 🆕 More examples

The power: the loop doesn't care how many shape types exist. Add a new `Rectangle` class tomorrow and the loop **needs no change**.

A second example: notifications. The sending code is written once.

In [7]:
class Notification:
    def __init__(self, message):
        self.message = message
    def send(self):
        print(f"Generic notice: {self.message}")

class Email(Notification):
    def send(self):
        print(f"📧 Emailing: {self.message}")

class SMS(Notification):
    def send(self):
        print(f"📱 Texting: {self.message[:20]}")      # SMS is short!

class PushAlert(Notification):
    def send(self):
        print(f"🔔 Pop-up: {self.message.upper()}")


for n in [Email("Your order shipped"), SMS("Your order shipped and arrives on Friday"), PushAlert("sale today")]:
    n.send()

📧 Emailing: Your order shipped
📱 Texting: Your order shipped a
🔔 Pop-up: SALE TODAY


---
## 4. Duck typing

> *"If it walks like a duck and quacks like a duck, then it is a duck."*

In Java or C#, polymorphism needs a shared parent class or an interface. In Python, an object only needs to **have the method you call**. The classes don't have to be related at all.

### 📚 From class

In [8]:
class Duck:
    def sound(self):
        print("Quack!")

class Dog:
    def sound(self):
        print("Woof!")

class Car:
    def sound(self):
        print("Vroom!")


def make_sound(thing):
    thing.sound()              # we don't check what it is, only that it can .sound()

for item in [Duck(), Dog(), Car()]:
    make_sound(item)

Quack!
Woof!
Vroom!


### 🆕 More examples

You have already used duck typing without noticing: `len()` works on strings, lists, tuples, dictionaries, and sets: anything that *knows its length*.

In [9]:
for thing in ["hello", [1, 2, 3], (4, 5), {"a": 1}, {7, 8, 9, 10}]:
    print(type(thing).__name__, len(thing))

str 5
list 3
tuple 2
dict 1
set 4


> ⚠️ The price of flexibility: if the object does *not* have the method, you only find out when the code runs.

In [10]:
make_sound("a string")

AttributeError: 'str' object has no attribute 'sound'

---
## 5. Overriding vs overloading

| | Overriding | Overloading |
|---|---|---|
| Meaning | a child **replaces** a parent's method | several methods, **same name, different parameters** |
| In Java / C# | ✅ | ✅ (natively) |
| In Python | ✅ | ❌ **not built in**: a later `def` with the same name just replaces the earlier one |

Python gets the same effect with **default parameters** and **`*args`**.

### 📚 From class

In [11]:
class Calculator:
    def add(self, a, b, c=0):        # c is optional: two OR three numbers
        return a + b + c

print(Calculator().add(5, 3))
print(Calculator().add(5, 3, 2))

8
10


In [12]:
class Calculator:
    def add(self, *args):             # any number of numbers
        return sum(args)

print(Calculator().add(5, 3))
print(Calculator().add(5, 3, 2, 5, 10, 21, 5.5, 10.2))

8
61.7


### 🆕 More examples

What really happens if you define the same method twice? The last one wins, and the first is gone:

In [13]:
class Greeter:
    def hello(self):
        return "Hello!"

    def hello(self, name):            # this REPLACES the first hello
        return f"Hello, {name}!"

g = Greeter()
print(g.hello("Sara"))
g.hello()                              # the zero-argument version no longer exists

Hello, Sara!


TypeError: Greeter.hello() missing 1 required positional argument: 'name'

---
## 6. `__eq__`: comparing objects

By default, `==` on two objects asks *"are they the very same object in memory?"* Two separate objects with identical data are **not** equal:

In [14]:
class Card:
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

a = Card("K", "hearts")
b = Card("K", "hearts")
print(a == b)           # False: two different objects
print(a is a)

False
True


Define **`__eq__`** to say what "equal" should mean for your class. It receives `other` (the thing on the right of `==`) and returns `True`/`False`. Always check that `other` is the right type first.

In [15]:
class Card:
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

    def __eq__(self, other):
        if not isinstance(other, Card):
            return False
        return self.rank == other.rank and self.suit == other.suit


a = Card("K", "hearts")
b = Card("K", "hearts")
c = Card("Q", "hearts")
print(a == b)           # True now
print(a == c)
print(a == "K hearts")  # different type: False, no crash

True
False
False


`__eq__` also makes `in` and `list.count` work on lists of objects:

In [16]:
hand = [Card("K", "hearts"), Card("2", "clubs")]
print(Card("2", "clubs") in hand)
print(Card("A", "spades") in hand)

True
False


> 💡 Other dunder methods you may meet: `__lt__` (`<`), `__add__` (`+`), `__len__` (`len()`), `__getitem__` (`obj[i]`). This is the "operator overloading" the Kaggle lesson talks about: it's why `+` can add numbers *and* join strings *and* add pandas columns.

---
## 7. `__repr__`: the developer's view

| | `__str__` | `__repr__` |
|---|---|---|
| For | **users** (friendly text) | **developers** (precise, debugging) |
| Used by | `print(obj)`, `str(obj)` | the notebook/console, `repr(obj)`, and **when the object is inside a list** |
| If missing | falls back to `__repr__` | `<__main__.X object at 0x...>` |

Good practice: make `__repr__` look like the code that would create the object.

In [17]:
class Pizza:
    def __init__(self, flavor, size_cm):
        self.flavor = flavor
        self.size_cm = size_cm

    def __str__(self):
        return f"{self.flavor} pizza ({self.size_cm} cm)"


orders = [Pizza("Margherita", 30), Pizza("BBQ", 35)]
print(orders[0])        # uses __str__
print(orders)           # a LIST shows each item's __repr__ ...

Margherita pizza (30 cm)
[<__main__.Pizza object at 0x7f88440f1010>, <__main__.Pizza object at 0x7f88440aa0d0>]


...which is the unhelpful default because we haven't written one. Add it:

In [18]:
class Pizza:
    def __init__(self, flavor, size_cm):
        self.flavor = flavor
        self.size_cm = size_cm

    def __str__(self):
        return f"{self.flavor} pizza ({self.size_cm} cm)"

    def __repr__(self):
        return f"Pizza(flavor={self.flavor!r}, size_cm={self.size_cm})"


orders = [Pizza("Margherita", 30), Pizza("BBQ", 35)]
print(orders[0])        # friendly: __str__
print(orders)           # precise: __repr__
Pizza("Pepperoni", 40)  # the last line of a cell is shown with __repr__

Margherita pizza (30 cm)
[Pizza(flavor='Margherita', size_cm=30), Pizza(flavor='BBQ', size_cm=35)]


Pizza(flavor='Pepperoni', size_cm=40)

`!r` inside an f-string means "use `repr()`": it puts quotes around strings so you can tell `'30'` from `30`.

> 💡 **Tip:** if you only have time to write one, write `__repr__`, because `print` falls back to it when `__str__` is missing.

---
## 8. `@property`: smarter getters and setters

In Part 1 we protected data with `get_size_cm()` and `set_size_cm()`. It works, but calling `pizza.set_size_cm(40)` feels clumsy compared with `pizza.size_cm = 40`.

**`@property`** gives you both: the **simple attribute syntax** on the outside, with **validation** on the inside.

- `@property` above a method → it becomes the **getter**, used like an attribute (no parentheses).
- `@name.setter` above a method with the same name → it becomes the **setter**.

### 📚 From class

In [19]:
class Pizza:
    def __init__(self, flavor, size_cm):
        self.flavor = flavor
        self.size_cm = size_cm            # looks like a plain assignment, but runs the setter!

    @property
    def size_cm(self):                    # getter
        return self._size_cm

    @size_cm.setter
    def size_cm(self, value):             # setter
        if not 15 <= value <= 50:
            raise ValueError("Pizza size must be between 15 and 50 cm.")
        self._size_cm = value


pizza = Pizza("Pepperoni", 30)
print(pizza.size_cm)        # the getter ran
pizza.size_cm = 40          # the setter ran
print(pizza.size_cm)

30
40


In [20]:
pizza.size_cm = 90          # the setter refuses

ValueError: Pizza size must be between 15 and 50 cm.

### 🆕 More examples

A property **without a setter** is **read-only**, perfect for values the outside world may look at but must not change. It can also be *calculated* on demand:

In [21]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):                       # calculated each time, always up to date
        return self.width * self.height


r = Rectangle(4, 5)
print(r.area)
r.width = 10
print(r.area)            # changed automatically
r.area = 99              # no setter: read-only

20
50


AttributeError: property 'area' of 'Rectangle' object has no setter

In [22]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Below absolute zero!")
        self._celsius = value

    @property
    def fahrenheit(self):                 # read-only, derived from celsius
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
print(t.fahrenheit)
t.celsius = 100
print(t.fahrenheit)

77.0
212.0


---
## 9. `isinstance()` and guard checks

`isinstance(obj, SomeClass)` answers *"is this object an instance of that class **or any of its children**?"*

| | `type(x) == Pizza` | `isinstance(x, MenuItem)` |
|---|---|---|
| Looks at | the **exact** class only | the class **and its parents** (inheritance-aware) |
| For a `Pizza` object and `MenuItem` | `False` | `True` ✅ |

Prefer `isinstance`. You can also pass a **tuple of types**: `isinstance(x, (int, float))`.

### 📚 From class

In [23]:
print(isinstance(5, int))
print(isinstance("hi", str))
print(isinstance(5, str))
print(isinstance(3.5, (int, float)))        # several types at once

True
True
False
True


In [24]:
class MenuItem:
    def __init__(self, name, price):
        self.name = name
        self.price = price

class Pizza(MenuItem):
    pass

class Drink(MenuItem):
    pass

margherita = Pizza("Margherita", 600)

print(isinstance(margherita, Pizza))        # True
print(isinstance(margherita, MenuItem))     # True: a Pizza IS-A MenuItem
print(isinstance(margherita, Drink))        # False
print(type(margherita) == MenuItem)         # False: type() ignores inheritance

True
True
False
False


### Guard functions
The most common use: **protect a function** from receiving the wrong kind of thing, and fail with a clear message instead of a confusing crash later.

In [25]:
def apply_discount(item, percent):
    if not isinstance(item, MenuItem):
        raise TypeError("Only menu items can be discounted.")
    return item.price * (100 - percent) / 100

print(apply_discount(margherita, 10))        # 540.0
apply_discount("pizza", 10)

540.0


TypeError: Only menu items can be discounted.

In [26]:
def add_tip(total, tip):
    if not isinstance(tip, (int, float)):
        raise TypeError("Tip must be a number.")
    return total + tip

print(add_tip(600, 50))
add_tip(600, "fifty")

650


TypeError: Tip must be a number.

**How to read a guard:** *"if it is NOT the right kind of thing → stop with a clear error. Otherwise carry on."*

### 🆕 Another use: treat different kinds differently

In [27]:
def describe(item):
    if isinstance(item, Pizza):
        return f"🍕 {item.name}"
    elif isinstance(item, Drink):
        return f"🥤 {item.name}"
    elif isinstance(item, MenuItem):
        return f"🍽️ {item.name}"
    return "not a menu item"

for thing in [Pizza("BBQ", 800), Drink("Cola", 150), MenuItem("Salad", 300), 42]:
    print(describe(thing))

🍕 BBQ
🥤 Cola
🍽️ Salad
not a menu item


> 💡 Notice the order: the **most specific** class (`Pizza`) is checked before the general one (`MenuItem`), otherwise a Pizza would match `MenuItem` first. If you find yourself writing long `isinstance` chains for behaviour, consider polymorphism (a method on each class) instead.

---
## 10. `@classmethod` and `@staticmethod`

Normal methods get `self` (one specific object). Two other kinds exist:

| Kind | First parameter | Can use… | Typical use |
|---|---|---|---|
| normal method | `self` | the object's data | most methods |
| **`@classmethod`** | `cls` (the class itself) | class data; can **create objects** | alternative constructors, e.g. `from_string` |
| **`@staticmethod`** | *(none)* | nothing from the class or object | a helper that belongs with the class, e.g. `is_valid_age(n)` |

> 💡 Think of it like this: **classmethod** = "I know which class I belong to." **staticmethod** = "I'm just a regular function that happens to live in the class's namespace."

These two are good to **recognise** when you read other people's code; you won't need them every day.

In [28]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_string(cls, text):               # builds a Person from "Ali-25"
        name, age = text.split("-")
        return cls(name, int(age))             # same as Person(name, int(age))

    @staticmethod
    def is_valid_age(age):                     # no self, no cls
        return 0 <= age <= 120

    def __str__(self):
        return f"{self.name} ({self.age})"


p = Person.from_string("Ali-25")               # called on the CLASS, no object needed
print(p)
print(Person.is_valid_age(150))
print(Person.is_valid_age(25))

Ali (25)
False
True


Why is `from_string` a **classmethod**? Because of `cls`: if a child class (like `Student(Person)`) calls it, `cls` is `Student`, so you get a `Student` back, not a plain `Person`.

In [29]:
class Student(Person):
    pass

s = Student.from_string("Sara-16")
print(type(s).__name__, s)

Student Sara (16)


**Another classmethod example, the one you'll see in the API notebook:** build an object from a dictionary (like JSON data):

In [30]:
class Pet:
    def __init__(self, name, species):
        self.name = name
        self.species = species

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data["species"])


raw = {"name": "Tom", "species": "cat"}
tom = Pet.from_dict(raw)
print(tom.name, tom.species)

Tom cat


---
## 11. Splitting code into files: modules and `__name__`

As programs grow, one giant file becomes hard to manage. Every `.py` file is a **module**, and you can `import` it from another file. This is how the Library project was split into `library.py` (the classes) and `main.py` (the menu).

Three ways to import:

```python
import pizza_shop                        # use as pizza_shop.Pizza(...)
from pizza_shop import Pizza, Drink      # use as Pizza(...)
import pizza_shop as ps                  # use as ps.Pizza(...)
```
### `if __name__ == "__main__":`
Every module has a built-in variable `__name__`:
- when you **run the file directly** → `__name__` is `"__main__"`
- when you **import it** from another file → `__name__` is the **module's name**

So this line means *"run this part only when the file is run directly, not when it's imported"*. It's the place for quick tests and demos.

Let's see it for real. The next cell **creates a file** called `pizza_shop.py` using the `%%writefile` magic.

In [31]:
%%writefile pizza_shop.py
print("pizza_shop.py is loading. __name__ is:", __name__)


class MenuItem:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __str__(self):
        return f"{self.name}: Rs {self.price}"


class Pizza(MenuItem):
    pass


class Drink(MenuItem):
    pass


if __name__ == "__main__":
    # quick self-test, runs only when this file is executed directly
    print("Running the self-test...")
    print(Pizza("Test pizza", 500))

Writing pizza_shop.py


**1. Import it.** The module's top-level code runs once, and `__name__` is the module's name. The test block is skipped:

In [32]:
import pizza_shop

print(pizza_shop.Pizza("Margherita", 600))
from pizza_shop import Drink
print(Drink("Cola", 150))

pizza_shop.py is loading. __name__ is: pizza_shop
Margherita: Rs 600
Cola: Rs 150


**2. Run it directly** (this is what happens when you type `python pizza_shop.py` in a terminal). Now `__name__` is `"__main__"` and the test block runs:

In [33]:
import subprocess, sys

result = subprocess.run([sys.executable, "pizza_shop.py"], capture_output=True, text=True)
print(result.stdout)

pizza_shop.py is loading. __name__ is: __main__
Running the self-test...
Test pizza: Rs 500



**3. In this notebook itself**, `__name__` is also `"__main__"`, because the notebook is the program being run:

In [34]:
print(__name__)

__main__


> ⚠️ **Common mistakes with modules**
> - The file you import must be in the **same folder** (or on Python's path).
> - Don't name your file the same as a standard module (`random.py`, `math.py`): your file would be imported instead of Python's!
> - Python caches imports. After editing a module in the same session, use `import importlib; importlib.reload(pizza_shop)` (or restart the kernel).
> - Avoid **circular imports** (A imports B and B imports A). Keep dependencies one-directional.

A tip: `module.__file__` shows where a module lives, which helps when you wonder *which* file Python really imported.

In [35]:
import random
print(random.__file__)          # a standard-library module
print(pizza_shop.__file__)      # our own module

/usr/local/lib/python3.13/random.py
/tmp/tmpkpr4z_sa/pizza_shop.py


Clean up the file we made:

In [36]:
import os, shutil, sys
sys.modules.pop("pizza_shop", None)
for path in ["pizza_shop.py"]:
    if os.path.exists(path):
        os.remove(path)
shutil.rmtree("__pycache__", ignore_errors=True)
print("cleaned up")

cleaned up


---
## 12. Practice problems

**🟢 Easy**
1. Make `Animal` with `name` and `speak()`. Create `Dog`, `Cat` and `Cow` that override `speak()`. Loop over a list of them.
2. Add `__repr__` to a class of your own and print a list of its objects.
3. Write a `Shape`/`Rectangle`/`Circle` family with `area()` and `perimeter()`.

**🟡 Medium**
4. Create `Account` with a `balance` **property** that refuses negative values. Create `SavingsAccount(Account)` with an `add_interest()` method.
5. Write `describe_all(items)` that takes a list and uses `isinstance` to print a different message for ints, strings, lists, and anything else.
6. Make a `Point` class with `__eq__`, `__repr__`, and a `distance_to(other)` method. Also add `Point.from_tuple((3, 4))` as a classmethod.
7. Create a `Library` of `Book`, `Magazine`, and `DVD` objects (all inheriting from `LibraryItem`), each with a different `loan_days()`. Print the loan days for each using one loop.

**🔴 Challenge**
8. **Text battle game:** `Character` with `health` and `attack()`; `Warrior`, `Mage`, `Archer` that override `attack()`. Two teams fight automatically until one team has no health left.
9. Split exercise 7 into `library.py` and `main.py`, with a menu in `main.py` and a self-test under `if __name__ == "__main__":` in `library.py`.

In [37]:
# your solutions here

---
### ✅ You can now…
- build class families with inheritance, `super()` and overriding
- explain polymorphism and duck typing (and overriding vs overloading)
- make objects comparable (`__eq__`) and printable (`__str__`, `__repr__`)
- use `@property` for validated, readable attributes
- guard functions with `isinstance`, and recognise `@classmethod` / `@staticmethod`
- split a program into modules with `if __name__ == "__main__":`

**Next notebook →** `08_APIs_with_requests.ipynb` (after reading the Developer Tools guide on `pip` and virtual environments)